In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from src.programme import load_talks
from src.geocode import attach_coordinates, geocode_affiliations
from src.plot_utils import (
    export_attendee_site_data,
    plot_affiliation_hexmap,
    plot_affiliation_map_interactive,
    plot_geocoding_summary,
)

talks = load_talks()
talks.head()

,talk_id,sid,title,presenter,primary_author,authors,affiliation,honorific,position,has_abstract,...,start,end,date,session_id,session_title,session_kind,session_code,session_theme,room,location
0,04769143-abef-4acf-b7e7-5bb9f45438cf,04769143,Being A Good Ancestor,Dan Hikuroa,Dan Hikuroa,[Dan Hikuroa],University of Auckland,<NA>,<NA>,True,...,09:00,09:45,2026-07-20,7f9919ea-7cc2-4195-b33e-42c0764b28ed,Plenary | Being A Good Ancestor,plenary,None,NaN,te-paepae,Te Paepae Theatre | Level 5 | NZICC
1,d4ed07a0-0e43-4168-beab-d96e34467813,d4ed07a0,Are Mesopredatory Sharks Functionally Redundan...,Mark Meekan,Mark Meekan,[Mark Meekan],University of Western Australia,Dr,Senior Research Fellow,True,...,10:15,10:30,2026-07-20,3dff507b-857f-49b3-a3d9-8896431d5090,Reef fish ecology and symbioses in a changing ...,session,1A,87.0,301,Meeting Room 301 | Level 3
2,ff36121e-e72a-45f2-9bed-40ecec669367,ff36121e,Temporal variations in the foraging behaviour ...,Thomas Johnstone,Thomas Johnstone,"[Thomas Johnstone, Navya Mittal, Will Rayment,...",Otago University,Mr,Student,True,...,10:30,10:45,2026-07-20,3dff507b-857f-49b3-a3d9-8896431d5090,Reef fish ecology and symbioses in a changing ...,session,1A,87.0,301,Meeting Room 301 | Level 3
3,03678681-6c39-476c-bd59-e9ef6adacc1d,03678681,Dietary partitioning of herbivorous fishes dif...,Madeleine Ward,Madeleine Ward,"[Madeleine Ward, Deron Burkepile, Thomas Adam,...",University of California - Santa Barbara,Ms,Phd Candidate,True,...,10:45,10:50,2026-07-20,3dff507b-857f-49b3-a3d9-8896431d5090,Reef fish ecology and symbioses in a changing ...,session,1A,87.0,301,Meeting Room 301 | Level 3
4,2334c0a7-a0a2-4366-bc68-6354988bbc87,2334c0a7,Prey diversity in the deep ocean: metabarcodin...,Diana Beltran Rodriguez,Diana Beltran Rodriguez,"[Stacey Williams, Carlos Prada, Diana Beltran ...",University of Rhode Island,Dr,Teaching Professor,True,...,10:50,10:55,2026-07-20,3dff507b-857f-49b3-a3d9-8896431d5090,Reef fish ecology and symbioses in a changing ...,session,1A,87.0,301,Meeting Room 301 | Level 3


In [3]:
talks.affiliation.str.contains("Micronesia").sum()


2

In [4]:
geocoded_affiliations = geocode_affiliations(
    talks["affiliation"].dropna().unique(),
    retry_failed=False,
)
talks_geo = attach_coordinates(talks, geocoded_affiliations)

geocoded_affiliations["geocoded"].value_counts()
# examples of failed geocodings
geocoded_affiliations[~geocoded_affiliations["geocoded"]].affiliation.unique()


Geocoding affiliations (646 unique, 640 cached, 6 overrides, 6 to query)

Output()

Done. Geocoded 0 | Failed 0 | Skipped 640 cached

array(['A*STAR Institute of High Performance Computing',
       'ACIAR Coral Restoration Governance Project',
       'Alfred Wegener Institute For Marine And Polar Research (awi)',
       'Alligator Head Foundation',
       'Aquatic Resources Research Institute, Chualalongkorn University',
       'Atlantic Oceanographic and Meteorological Laboratory',
       'Blue Corner Marine Research',
       'C2o Consulting And James Cook University', 'CMOANA Consulting',
       'CNRS/UPVD', 'CSIR National Institute of Oceanography',
       'CSS-INC., Under Contract to NOAA-NCCOS',
       'Cape Eleuthera Institute',
       'Center For Research And Advanced Studies Ipn-mérida Unit',
       'Center for Marine and Environmental Studies',
       'Centre for Scientific Research and Higher Education of Ensenada',
       'Centro de Ciências do Mar do Algarve',
       'Centro de Investigaciones Biológicas del Noroeste',
       'Charles Darwin Foundation', 'Citizens of the Reef',
       'Conservation Metric

In [5]:
talks_geo[talks_geo.presenter.str.contains("Heron")]


,talk_id,sid,title,presenter,primary_author,authors,affiliation,honorific,position,has_abstract,...,session_kind,session_code,session_theme,room,location,latitude,longitude,geocoded,geocode_level,query_used
1148,6281686b-d319-462f-9d91-583c6f76c2aa,6281686b,"Pluralising ""art"" – the role of performance ar...",Scott Heron,Scott Heron,[Scott Heron],James Cook University,Prof,Professor,True,...,session,3C,61.0,518,Meeting Room 518 | Level 5,-19.328962,146.756645,True,None,None
1957,04304b1e-824f-4ccd-ae7c-6b77af4a8378,04304b1e,"Longer, higher, broader – multidimensional rec...",Scott Heron,Scott Heron,"[Scott Heron, Blake Spady, Jessica Benthuysen,...",James Cook University,Prof,Professor,True,...,session,5A,116.0,518,Meeting Room 518 | Level 5,-19.328962,146.756645,True,None,None
1970,fff29f4c-6fe8-4d4e-8e55-4e66ea8d5d97,fff29f4c,Evaluating the past decade of heat stress on W...,Scott Heron,Scott Heron,"[Scott Heron, Jon Day, Blake Spady]",James Cook University,Prof,Professor,True,...,session,5A,130.0,te-paepae,Te Paepae Theatre | Level 5 | NZICC,-19.328962,146.756645,True,None,None


In [8]:
"dan" in str(talks_geo.authors[0]).lower()


True

In [ ]:
from collections import Counter

name = "miguel mies"  # partial match, case-insensitive


def matching_author_names(authors_series, query: str) -> set[str]:
    query = query.lower().strip()
    names: set[str] = set()
    for authors in authors_series.dropna():
        if not isinstance(authors, list):
            continue
        for author in authors:
            if query in author.lower():
                names.add(author)
    return names


def talks_for_authors(df, author_names: set[str]):
    return df[
        df["authors"].apply(
            lambda authors: (
                isinstance(authors, list) and bool(set(authors) & author_names)
            )
        )
    ]


def coauthorship_connections(df, author_names: set[str]) -> Counter:
    """Same metric as the network tab: co-authorship link weights."""
    degrees: Counter = Counter()
    for authors in df["authors"].dropna():
        if not isinstance(authors, list):
            continue
        matched = [author for author in authors if author in author_names]
        if not matched:
            continue
        for person in matched:
            for coauthor in authors:
                if coauthor != person:
                    degrees[person] += 1
    return degrees


author_names = matching_author_names(talks_geo["authors"], name)
matches = talks_for_authors(talks_geo, author_names)
connections = coauthorship_connections(talks_geo, author_names)

print("Matched author names:", sorted(author_names, key=str.casefold))
print("Talks:", len(matches))
print("Co-authorship connections:", dict(connections))
print("(Network tab 'connections' = co-authorship links, not talk count)")

matches[["presenter", "title", "authors"]]

18

In [ ]:
# plot_geocoding_summary(geocoded_affiliations)
# plt.show()

# fig, ax = plot_affiliation_hexmap(
#     talks_geo,
#     save_path="outputs/speaker_affiliation_hexmap.png",
#     h3_resolution=10,
# )
# plt.show()

interactive_map = plot_affiliation_map_interactive(
    talks_geo,
    save_path="outputs/speaker_affiliation_map.html",
)
interactive_map

site_data_path = export_attendee_site_data(
    talks_geo,
    save_path="js/locations.js",
)
site_data_path

In [ ]:
attendee_map_path = export_attendee_map_html(
    talks_geo,
    save_path="outputs/attendee_map.html",
)
